In [1]:
import pandas as pd
import numpy as np
import os
import re

In [2]:
sim_type = 'clinvar'
sensitivity = ''
padding = 0
cn_values = [1, 3]

In [3]:
run_number = '29'
work_folder_extension = ''
dnascreen_run = run_number + work_folder_extension
sim_dir='/Users/zlaw0001/Library/CloudStorage/OneDrive-MonashUniversity/Joshua_PhD_Project/Aim_1/CNV_calling/simulation_hom_deletion/experiment_onsimulation_using_clinvar'
exome_depth_dir='/Users/zlaw0001/Library/CloudStorage/OneDrive-MonashUniversity/Joshua_PhD_Project/Aim_1/CNV_calling/exome_depth'

In [4]:
selected_windows_dir = sim_dir + '/run' + dnascreen_run + '/selected_regions'

In [5]:
# Load the bed file into a dataframe
exons = pd.read_csv(exome_depth_dir + '/bed_files/9genes_25bp.fix.sorted.bed', sep='\t', header=None, names=['chr', 'start', 'end', 'name', 'idk','strand'])

## Checking exons covered by selected_windows

In [6]:
# Check if the boundary of the simulated window falls within an exon for both VS and ED!

results = []

for cn in cn_values:
    selected_windows_file = selected_windows_dir + f'/selected_regions_cn{cn}_sample'
    selected_windows = pd.read_csv(selected_windows_file, sep='\t', 
                               header=None,
                               names=['sample', 'chr', 'start', 'end', 'gene'])
    
    # Doing this because we have repeated samples with different CN
    selected_windows['sample'] = 'cn' + str(cn) + '_' + selected_windows['sample'] + '.sorted'
    
    # Iterate over each row(sample) in the selected_windows dataframe
    for index, window in selected_windows.iterrows():
        # Get the chromosome, start, and end of the current window
        chr_window = window['chr']
        start_window = window['start']
        end_window = window['end']
        
        # Filter exons to find those that are on the same chromosome and within or overlap with the start and end of the current window
        exons_overlapping_window = exons[(exons['chr'] == chr_window) &
                                (exons['start'] < end_window) &
                                (exons['end'] > start_window)]
        
        # Count the number of exons that match the criteria
        count_exons_overlapping_simulated_window = len(exons_overlapping_window)

        # Filter exons to find those that are on the same chromosome and within the start and end of the current window
        exons_within_window = exons[(exons['chr'] == chr_window) &
                                (exons['start'] >= start_window) &
                                (exons['end'] <= end_window)]
        
        # Count the number of exons that match the criteria
        count_exons_within_simulated_window = len(exons_within_window)

        
        # Calculate the summed length of the exons
        summed_length_exons_overlapping = (exons_overlapping_window['end'] - exons_overlapping_window['start']).sum()
        summed_length_exons_within = (exons_within_window['end'] - exons_within_window['start']).sum()
        
        # Check if the start_window or end_window falls within any exons
        overlap_with_start = exons[(exons['chr'] == chr_window) &
                                   (exons['start'] <= start_window) &
                                   (exons['end'] >= start_window)]

        overlap_with_end = exons[(exons['chr'] == chr_window) &
                                 (exons['start'] <= end_window) &
                                 (exons['end'] >= end_window)]

        NA_dict = {'chr':'NA', 'start':'NA', 'end':'NA', 'name':'NA'}

        # Convert to list of dictionaries with required fields or 'NA' if no overlap
        overlap_with_start_dict = (overlap_with_start[['chr', 'start', 'end', 'name']]
                                   .drop_duplicates()
                                   .to_dict(orient='records')[0] if not overlap_with_start.empty else NA_dict)

        overlap_with_end_dict = (overlap_with_end[['chr', 'start', 'end', 'name']]
                                 .drop_duplicates()
                                 .to_dict(orient='records')[0] if not overlap_with_end.empty else NA_dict)
        
        # Append the result to the results list
        results.append({
            'cn': cn,
            'sample': window['sample'],
            'chr': chr_window,
            'start': start_window,
            'end': end_window,
            'gene': window['gene'],
            'count_exons_overlapping_simulated_window': count_exons_overlapping_simulated_window,
            'count_exons_within_simulated_window': count_exons_within_simulated_window,
            'summed_length_exons_overlapping': summed_length_exons_overlapping,
            'summed_length_exons_within':summed_length_exons_within,
            'overlap_with_start_chr': overlap_with_start_dict['chr'],
            'overlap_with_start_start': overlap_with_start_dict['start'],
            'overlap_with_start_end': overlap_with_start_dict['end'],
            'overlap_with_start_name': overlap_with_start_dict['name'],
            'overlap_with_end_chr': overlap_with_end_dict['chr'],
            'overlap_with_end_start': overlap_with_end_dict['start'],
            'overlap_with_end_end': overlap_with_end_dict['end'],
            'overlap_with_end_name': overlap_with_end_dict['name'],
        })


In [7]:
overlapping_exons = pd.DataFrame(results)

In [8]:
overlapping_exons.to_csv(sim_dir + '/run' + dnascreen_run + '/plot_generation/simulated_windows_info.csv')

In [9]:
overlapping_exons

,cn,sample,chr,start,end,gene,count_exons_overlapping_simulated_window,count_exons_within_simulated_window,summed_length_exons_overlapping,summed_length_exons_within,overlap_with_start_chr,overlap_with_start_start,overlap_with_start_end,overlap_with_start_name,overlap_with_end_chr,overlap_with_end_start,overlap_with_end_end,overlap_with_end_name
0,1,cn1_DNS-XTHS-0105-D03-DNS009757_S20_chr2_21001...,chr2,21001710,21035728,APOB,23,21,14149,12319,chr2,21001704,21003359,APOB_cds_0,chr2,21035558,21035733,APOB_cds_22
1,1,cn1_DNS-XTHS-0107-E04-DNS009958_S221_chr2_4740...,chr2,47403067,47483228,MSH2,16,16,3605,3605,NA,NA,NA,NA,NA,NA,NA,NA
2,1,cn1_DNS-XTHS-0106-C09-DNS009900_S163_chr2_4740...,chr2,47403067,47408555,MSH2,2,1,466,261,NA,NA,NA,NA,chr2,47408375,47408580,MSH2_cds_1
3,1,cn1_DNS-XTHS-0107-F08-DNS009991_S254_chr2_4740...,chr2,47403067,47403402,MSH2,1,0,261,0,NA,NA,NA,NA,chr2,47403166,47403427,MSH2_cds_0
4,1,cn1_DNS-XTHS-0105-E07-DNS009790_S53_chr2_47403...,chr2,47403067,47429741,MSH2,7,6,1626,1376,NA,NA,NA,NA,chr2,47429716,47429966,MSH2_cds_6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
621,3,cn3_DNS-XTHS-0107-G08-DNS009992_S255_chr19_111...,chr19,11116092,11116999,LDLR,2,0,359,0,chr19,11116068,11116237,LDLR_cds_10,chr19,11116833,11117023,LDLR_cds_11
622,3,cn3_DNS-XTHS-0106-H04-DNS009865_S128_chr19_111...,chr19,11120082,11129680,LDLR,5,3,952,552,chr19,11120066,11120258,LDLR_cds_12,chr19,11129487,11129695,LDLR_cds_16
623,3,cn3_DNS-XTHS-0107-E05-DNS009966_S229_chr19_111...,chr19,11120092,11129670,LDLR,5,3,952,552,chr19,11120066,11120258,LDLR_cds_12,chr19,11129487,11129695,LDLR_cds_16
624,3,cn3_DNS-XTHS-0105-A09-DNS009802_S65_chr19_1112...,chr19,11120092,11120522,LDLR,2,0,395,0,chr19,11120066,11120258,LDLR_cds_12,chr19,11120344,11120547,LDLR_cds_13


## Checking for number of CNVs that do not have an exon completely within the CNV coordinates.

In [10]:
print(str(len(overlapping_exons[overlapping_exons['count_exons_within_simulated_window'] == 0])) + ' CNVs that do not have an exon completely within the CNV coordinates.')

315 CNVs that do not have an exon completely within the CNV coordinates.


## Storing per sample CNV calls for varseq

In [11]:
varseq_tables_path = sim_dir + '/run' + dnascreen_run + '/plot_generation/varseq_tables' + sensitivity
    
cnvcalls_path = os.path.join(varseq_tables_path, 'all_cnv_calls_sim_clinvar.tsv')
sample_path = os.path.join(varseq_tables_path, 'Varseq_sample_sim_clinvar.tsv')

cnvcalls_df = pd.read_csv(cnvcalls_path, sep='\t')
sample_df = pd.read_csv(sample_path, sep = '\t')

cov_sim_sample_dir = os.path.join(varseq_tables_path, 'cov_samples')

/var/folders/hh/8vzwcgj52x12km60yd3vghr0213gzj/T/ipykernel_80865/633117776.py:6: DtypeWarning: Columns (31,42,43,44,54,55,66,67,78,91,102,103,114,115,126,127,138,139,150,151,152,162,163,174,187,198,199,210,223,234,235,236,246,259,270,271,282,283,294,295,296,306,307,318,319,330,331,342,343,344,354,355,366,367,378,379,390,391,402,403,414,415,426,427,428,438,439,440,450,451,462,463,474,475,476,486,499,510,511,522,523,534,535,546,547,558,559,560,570,571,582,583,594,595,606,607,618,619,630,631,642,643,654,667,678,679,680,690,703,704,714,715,726,727,738,739,750,751,762,763,774,775,786,787,798,799,810,811,822,823,834,835,846,847,848,858,859,870,871,872,882,895,896,906,907,918,919,930,931,942,943,954,955,966,979,990,991,1002,1003,1004,1014,1015,1026,1027,1038,1051,1062,1063,1074,1075,1086,1087,1098,1099,1110,1111,1112,1122,1123,1134,1159,1170,1171,1182,1195,1206,1207,1218,1231,1242,1243,1244,1254,1255,1266,1267,1278,1279,1280,1290,1291,1302,1303,1314,1315,1326,1327,1338,1339,1350,1351,1362,136

## The code below is a sanity check of whether samples in reference set are unique and not repeated. Also checks if the target sample is in the reference set itself. 

In [12]:
# Initialize counters and lists to store row indices
duplicate_rows = []
matching_rows = []

# Process each row in sample_df
for index, row in sample_df.iterrows():
    # Get main sample ID without prefix 'cn#_' and suffix '.sorted'
    main_sample_id = re.sub(r'^(cn\d+_)?|\.sorted$', '', row['Samples'])

    # Initialize lists to hold processed reference samples
    sample_names = []

    # Split the reference samples by commas
    for item in row['Reference Samples'].split(','):
        # Extract the sample name using regex
        name_match = re.search(r'^[^\(]+', item)
        
        if name_match:
            # Clean the reference sample name by removing '.hq.sorted.marked' and '.sorted'
            ref_sample_id = re.sub(r'\.hq\.sorted\.marked|\.sorted$', '', name_match.group().strip())
            sample_names.append(ref_sample_id)

    # Check for duplicate reference samples
    if len(sample_names) != len(set(sample_names)):
        duplicate_rows.append(index)  # Store the index of the row with duplicates

    # Check if any reference sample matches the main sample ID
    if main_sample_id in sample_names:
        matching_rows.append(index)  # Store the index of the row with matches

# Print the results
if duplicate_rows:
    print(f"Number of rows with duplicate reference samples: {len(duplicate_rows)}")
    print(f"Rows with duplicates: {duplicate_rows}")
else:
    print("No rows have duplicate reference samples.")

if matching_rows:
    print(f"Number of rows with reference samples matching main sample ID: {len(matching_rows)}")
    print(f"Rows with matches: {matching_rows}")
else:
    print("No rows have reference samples matching main sample ID.")

No rows have duplicate reference samples.
No rows have reference samples matching main sample ID.


## The code below extracts CNV calls per sample.

In [13]:
real_varseq_data_dir = "/Users/zlaw0001/Library/CloudStorage/OneDrive-MonashUniversity/Joshua_PhD_Project/Aim_1/CNV_calling/VarSeq_CNV/varseq_tables_on_real_data" + "/run" + run_number
cov_real_sample_dir = real_varseq_data_dir + "/cov_samples"

In [14]:
import numpy as np
import pandas as pd
import os

# Storing average z-scores of non-matching exons for each copy number (CN)
avg_z_scores_all_nonmatching_sim_exons = [[] for _ in range(5)]

# Lists to store sample IDs, metrics for CNV calls, and metrics for coverage regions if the CNV is missed
sample_lst = []
percent_diff_lst = []
avg_nonmatching_sim_z_score_lst = []
CNV_coord_chr = []
CNV_coord_start = []
CNV_coord_end = []
region_size_lst = []
number_of_exons_varseq = []
type_lst = []
flag_lst = []
orig_target_mean_depth_lst = []
orig_z_score_lst = []
orig_ratio_lst = []
sim_target_mean_depth_lst = []
sim_z_score_lst = []
sim_ratio_lst = []
VS_target_mean_depth_lst = []
VS_z_score_lst = []
VS_ratio_lst = []
variants_considered_lst = []
supporting_LOH_variants_lst = []
estimated_CN_lst = []
GC_content_lst = []
p_val_lst = []
precision_levels_lst = []

# Iterate over each sample
for i in range(31, len(cnvcalls_df.columns), 12):
    sample_id = cnvcalls_df.columns[i][:-10]

    # Extract real sample id using re.search
    sample_id_real = re.search(r"(DNS-[A-Z0-9-]+_S[0-9]+)", sample_id).group(0)
    
    sample_cn = overlapping_exons[overlapping_exons['sample'] == sample_id]['cn'].values[0]
    percent_diff = sample_df[sample_df['Samples'] == sample_id]['Percent Difference'].values[0]
    
    # Load simulated coverage statistics
    cov_sim_sample_file = os.path.join(cov_sim_sample_dir, f'cov_sim_{sim_type} - {sample_id}.tsv')
    cov_sim_sample_df = pd.read_csv(cov_sim_sample_file, sep='\t')

    # Load original coverage statistics
    cov_real_sample_file = os.path.join(cov_real_sample_dir, f'CNV_run{run_number} - {sample_id_real}.tsv')
    cov_real_sample_df = pd.read_csv(cov_real_sample_file, sep='\t')

    # Check if there is a CNV call in VarSeq with p-value <= 0.05
    any_value_check = cnvcalls_df.iloc[0:len(cnvcalls_df), i+10].apply(
        lambda x: not pd.isnull(x) and (float(x) < 0.05)
    )

    # Define simulated CNV coordinates
    sim_chr = int(overlapping_exons.loc[overlapping_exons['sample'] == sample_id]['chr'].values[0][3:])
    sim_start = overlapping_exons.loc[overlapping_exons['sample'] == sample_id]['start'].values[0]
    sim_end = overlapping_exons.loc[overlapping_exons['sample'] == sample_id]['end'].values[0]
    # count_exons_overlapping_simulated_window = overlapping_exons.loc[overlapping_exons['sample'] == sample_id]['count_exons_overlapping_simulated_window'].values[0]

    # Extract chromosome, start, and end from the Region column
    cov_sim_sample_df[['chr', 'start', 'end']] = cov_sim_sample_df['Region'].str.extract(r'(\d+):(\d+)-(\d+)')
    cov_sim_sample_df['chr'] = cov_sim_sample_df['chr'].astype(int)
    cov_sim_sample_df['start'] = cov_sim_sample_df['start'].astype(int)
    cov_sim_sample_df['end'] = cov_sim_sample_df['end'].astype(int)

    mask = (
        (cov_sim_sample_df['chr'] == sim_chr) &
        # checking for exon intervals that overlap or are within a simulated CNV
        ((((cov_sim_sample_df['start'] <= sim_end) &
         (cov_sim_sample_df['start'] >= sim_start)) |
        ((cov_sim_sample_df['end'] <= sim_end) &
         (cov_sim_sample_df['end'] >= sim_start))) | 
         # for simulated CNVs that are smaller than an exon interval
        (((cov_sim_sample_df['start'] <= sim_end) &
         (cov_sim_sample_df['start'] <= sim_start)) &
        ((cov_sim_sample_df['end'] >= sim_end) &
         (cov_sim_sample_df['end'] >= sim_start))))
    )

    cov_real_sample_df[['chr', 'start', 'end']] = cov_real_sample_df['Region'].str.extract(r'(\d+):(\d+)-(\d+)')
    cov_real_sample_df['chr'] = cov_real_sample_df['chr'].astype(int)
    cov_real_sample_df['start'] = cov_real_sample_df['start'].astype(int)
    cov_real_sample_df['end'] = cov_real_sample_df['end'].astype(int)

    mask = (
        (cov_real_sample_df['chr'] == sim_chr) &
        # checking for exon intervals that overlap or are within a simulated CNV
        ((((cov_real_sample_df['start'] <= sim_end) &
         (cov_real_sample_df['start'] >= sim_start)) |
        ((cov_real_sample_df['end'] <= sim_end) &
         (cov_real_sample_df['end'] >= sim_start))) | 
         # for simulated CNVs that are smaller than an exon interval
        (((cov_real_sample_df['start'] <= sim_end) &
         (cov_real_sample_df['start'] <= sim_start)) &
        ((cov_real_sample_df['end'] >= sim_end) &
         (cov_real_sample_df['end'] >= sim_start))))
    )


    # Extract non-matching exons' Z-Score and calculate average Z-score
    non_matching_sim_exons = cov_sim_sample_df[~mask].drop(columns=['chr', 'start', 'end'])
    z_score_non_matching_sim_exons = non_matching_sim_exons[f'Z Score for {sample_id}'].values
    avg_nonmatching_z_score_sim = np.mean(z_score_non_matching_sim_exons)
    avg_z_scores_all_nonmatching_sim_exons[sample_cn].extend(z_score_non_matching_sim_exons)


    # Extract metrics for matching exons in the simulated data
    matching_exons_sim = cov_sim_sample_df[mask].drop(columns=['chr', 'start', 'end'])
    mean_depth_sim = np.mean(matching_exons_sim[f'{sample_id} Mean Depth'])
    z_score_sim = np.mean(matching_exons_sim[f'Z Score for {sample_id}'])
    ratio_sim = np.mean(matching_exons_sim[f'Ratio for {sample_id}'])
    variants_considered = sum(matching_exons_sim[f'Variants Considered for {sample_id}'])

    # Extract metrics for matching exons in the original data
    matching_exons_orig = cov_real_sample_df[mask].drop(columns=['chr', 'start', 'end'])
    mean_depth_orig = np.mean(matching_exons_orig[f'{sample_id_real} Mean Depth'])
    z_score_orig = np.mean(matching_exons_orig[f'Z Score for {sample_id_real}'])
    ratio_orig = np.mean(matching_exons_orig[f'Ratio for {sample_id_real}'])
    # variants_considered = sum(matching_exons_sim[f'Variants Considered for {sample_id_real}'])

    if any(any_value_check):
        for v in range(0, len(any_value_check)):
            if any_value_check.iloc[v]:

                # Append sample-specific metrics
                sample_lst.append(sample_id)
                percent_diff_lst.append(percent_diff)
                avg_nonmatching_sim_z_score_lst.append(avg_nonmatching_z_score_sim)

                # Extract chr, start, and end of VarSeq calls
                chr = 'chr' + cnvcalls_df.iloc[v, 0].split(':')[0]
                start_end = cnvcalls_df.iloc[v, 0].split(':')[1]
                start = start_end.split('-')[0]
                end = start_end.split('-')[1]
                
                CNV_coord_chr.append(chr)
                CNV_coord_start.append(start)
                CNV_coord_end.append(end)
                region_size_lst.append(cnvcalls_df.iloc[v, 4])
                number_of_exons_varseq.append(cnvcalls_df.iloc[v, 2])
                type_lst.append(cnvcalls_df.iloc[v, i])
                flag_lst.append(cnvcalls_df.iloc[v, i+1])

                # Simulated region metrics
                sim_target_mean_depth_lst.append(mean_depth_sim)
                sim_z_score_lst.append(z_score_sim)
                sim_ratio_lst.append(ratio_sim)

                # Original region metrics
                orig_target_mean_depth_lst.append(mean_depth_orig)
                orig_z_score_lst.append(z_score_orig)
                orig_ratio_lst.append(ratio_orig)

                # VarSeq metrics
                VS_target_mean_depth_lst.append(cnvcalls_df.iloc[v, i+2])
                VS_z_score_lst.append(cnvcalls_df.iloc[v, i+3])
                VS_ratio_lst.append(cnvcalls_df.iloc[v, i+4])
                variants_considered_lst.append(variants_considered)
                supporting_LOH_variants_lst.append(cnvcalls_df.iloc[v, i+7])
                estimated_CN_lst.append(cnvcalls_df.iloc[v, i+5])
                GC_content_lst.append(cnvcalls_df.iloc[v, i+9])
                p_val_lst.append(cnvcalls_df.iloc[v, i+10])
                precision_levels_lst.append(cnvcalls_df.iloc[v, i+11])
    else:
        # Append metrics if no CNV call was made
        sample_lst.append(sample_id)
        percent_diff_lst.append(percent_diff)
        avg_nonmatching_sim_z_score_lst.append(avg_nonmatching_z_score_sim)

        CNV_coord_chr.append(np.nan)
        CNV_coord_start.append(np.nan)
        CNV_coord_end.append(np.nan)
        region_size_lst.append(np.nan)
        number_of_exons_varseq.append(np.nan)
        type_lst.append(np.nan)
        flag_lst.append(np.nan)

        # Simulated region metrics
        sim_target_mean_depth_lst.append(mean_depth_sim)
        sim_z_score_lst.append(z_score_sim)
        sim_ratio_lst.append(ratio_sim)

        # Original region metrics
        orig_target_mean_depth_lst.append(mean_depth_orig)
        orig_z_score_lst.append(z_score_orig)
        orig_ratio_lst.append(ratio_orig)

        # VarSeq metrics
        VS_target_mean_depth_lst.append(np.nan)
        VS_z_score_lst.append(np.nan)
        VS_ratio_lst.append(np.nan)
        variants_considered_lst.append(variants_considered)
        supporting_LOH_variants_lst.append(np.nan)
        estimated_CN_lst.append(np.nan)
        GC_content_lst.append(np.nan)
        p_val_lst.append(np.nan)
        precision_levels_lst.append(np.nan)


In [15]:
df_sample_CNV = pd.DataFrame({'sample':sample_lst, 
                              'Percent Difference':percent_diff_lst,
                              'Avg Z-score of Non-matching Exons':avg_nonmatching_sim_z_score_lst,
                              'VS_Call_chr':CNV_coord_chr,
                              'VS_Call_start':CNV_coord_start, 
                              'VS_Call_end':CNV_coord_end, 
                              'Size of CNV': region_size_lst,
                              'Number of Exons by VarSeq call':number_of_exons_varseq,
                                'Type of CNV':type_lst,
                              'VarSeq Flags for CNV': flag_lst,
                              'Target Mean Depth of Original CNV region': orig_target_mean_depth_lst,
                             'Z-Score of Original CNV region': orig_z_score_lst,
                             'Read Ratio of Original CNV region': orig_ratio_lst,
                              'Target Mean Depth of Simulated CNV': sim_target_mean_depth_lst,
                             'Z-Score of Simulated CNV': sim_z_score_lst,
                             'Read Ratio of Simulated CNV': sim_ratio_lst,
                             'VS Target Mean Depth of CNV': VS_target_mean_depth_lst,
                             'VS Z-Score of CNV': VS_z_score_lst,
                             'VS Read Ratio of CNV': VS_ratio_lst,
                              'Variants considered in CNV': variants_considered_lst,
                              'Supporting LOH variants in CNV': supporting_LOH_variants_lst,
                             'Estimated CN of CNV': estimated_CN_lst,
                             'GC Content of CNV': GC_content_lst,
                             'p-value of CNV':p_val_lst,
                             'Precision Level of call:':precision_levels_lst}
                            )

In [16]:
# Find duplicate 'Sample' values. These are false positives since we only simulated one CNV per sample.
duplicate_samples = df_sample_CNV['sample'][df_sample_CNV['sample'].duplicated(keep=False)]

# Print or store the duplicate samples
print(duplicate_samples)

2      cn1_DNS-XTHS-0105-A02-DNS009746_S9_chr17_43045...
3      cn1_DNS-XTHS-0105-A02-DNS009746_S9_chr17_43045...
4      cn1_DNS-XTHS-0105-A02-DNS009746_S9_chr19_11100...
5      cn1_DNS-XTHS-0105-A02-DNS009746_S9_chr19_11100...
72     cn1_DNS-XTHS-0105-D02-DNS009749_S12_chr3_36993...
                             ...                        
656    cn3_DNS-XTHS-0107-E05-DNS009966_S229_chr19_111...
658    cn3_DNS-XTHS-0107-E11-DNS010014_S277_chr13_323...
659    cn3_DNS-XTHS-0107-E11-DNS010014_S277_chr13_323...
660    cn3_DNS-XTHS-0107-F08-DNS009991_S254_chr2_4742...
661    cn3_DNS-XTHS-0107-F08-DNS009991_S254_chr2_4742...
Name: sample, Length: 84, dtype: object


In [17]:
VarSeq_results_df  = overlapping_exons.merge(df_sample_CNV, on='sample', how='left')

In [18]:
len(np.unique(VarSeq_results_df['sample']))

626

## Calculating the overlap percentage between the CNVs called and the simulated windows.

## The percentage is calculated based on the length of the 'Called CNV', and also the 'Simulated CNV' being the denominator

In [19]:
# Outputting results of overlapping and percentage of CNV call that is within the simulated window
# we know some simulated windows have their boundaries fall within exons.
# So the CNV coordinates called won't be entirely within the simulated windows as the approach is a 
# Read-Depth based approach.

# Function to calculate overlap percentage
def calculate_within_window_percentage(start1, end1, start2, end2, denominator):
    overlap_start = max(start1, start2)
    overlap_end = min(end1, end2)
    overlap_length = max(0, overlap_end - overlap_start)
    cnvcall_length = end2 - start2
    window_length = end1 - start1
    if denominator == 'call length':
        return overlap_length / cnvcall_length if cnvcall_length > 0 else 0
    elif denominator == 'window length':
        return overlap_length / window_length if window_length > 0 else 0

# Iterate over the rows and calculate the overlap
def output_overlapping_and_within_window_percentage(row, denominator):
    # Extract chr, start, end from CNV_coordinates
    if not pd.isna(row['VS_Call_chr']):
        cnv_chr = int(row['VS_Call_chr'][3:])
        cnv_start = int(row['VS_Call_start'])
        cnv_end = int(row['VS_Call_end'])
        
        if cnv_chr != int(row['chr'][3:]):
            return 0
        else:
            overlap_percentage = calculate_within_window_percentage(row['start'], 
                                                                    row['end'], cnv_start, cnv_end, denominator)
            return overlap_percentage
    else:
        return np.nan

# Apply the function with the specified denominator
VarSeq_results_df['VarSeq_overlap_within_window_based_on_call_length'] = VarSeq_results_df.apply(
    lambda row: output_overlapping_and_within_window_percentage(row, "call length"), axis=1
)

VarSeq_results_df['VarSeq_overlap_within_window_based_on_window_length'] = VarSeq_results_df.apply(
    lambda row: output_overlapping_and_within_window_percentage(row, "window length"), axis=1
)

## Storing per sample RECALLED CNV calls for each sample-simulated CNV pair
### A ClinVar CNV is considered recalled if there is a CNV called that overlaps the ClinVar CNV coordinates (overlap > 0).
### We will have a separate table 'recalled_cnvs_not_all_exons_called' for recalled CNVs that do not have all exons within CNV that are detected

In [20]:
# Function to count simulated exons within the VS-called coordinates
def counting_sim_exons_within_VS_call(VS_chr, VS_start, VS_end, sim_chr, sim_start, sim_end, exons):
    # Check if any of the inputs are NaN; if so, return 0 as the count
    if pd.isna(VS_chr) or pd.isna(VS_start) or pd.isna(VS_end) or pd.isna(sim_chr) or pd.isna(sim_start) or pd.isna(sim_end):
        return 0
    
    # Ensure VS_chr and sim_chr are strings and coordinates are integers
    VS_chr = str(VS_chr)
    VS_start = int(VS_start)
    VS_end = int(VS_end)
    sim_chr = str(sim_chr)
    sim_start = int(sim_start)
    sim_end = int(sim_end)
    
    # Filter exons within simulated chromosome and coordinates
    exons_on_sim_chr = exons[(exons['chr'] == sim_chr) & (exons['start'] >= sim_start) & (exons['end'] <= sim_end)]
    
    # From the filtered exons, count how many are fully contained within VS_start and VS_end
    # +1 adjustment for 
    count = sum((exons_on_sim_chr['start']+1 >= VS_start) & (exons_on_sim_chr['end'] <= VS_end))
    return count

# Function to count all exons within the VS-called coordinates
def counting_all_exons_overlapping_VS_call(VS_chr, VS_start, VS_end, exons):
    # Check if any of the inputs are NaN; if so, return 0 as the count
    if pd.isna(VS_chr) or pd.isna(VS_start) or pd.isna(VS_end):
        return 0
    
    # Ensure VS_chr and sim_chr are strings and coordinates are integers
    VS_chr = str(VS_chr)
    VS_start = int(VS_start)
    VS_end = int(VS_end)
    
    # Filter exons within simulated chromosome and coordinates
    exons_on_VS_chr = exons[(exons['chr'] == VS_chr) & (exons['start']+1 >= VS_start) & (exons['end'] <= VS_end)]
    
    # From the filtered exons, count how many are fully contained within VS_start and VS_end
    # +1 adjustment for 
    count = len(exons_on_VS_chr)
    return count


# Function to calculate percentage of total length of all exons that are within
# and/or partially overlapping with the ClinVar CNV coordinate
def calculate_exon_overlap_percentage(row, exons_df):
    # Filter exons on the same chromosome and within or overlapping the CNV coordinates
    overlapping_exons = exons_df[
        (exons_df['chr'] == row['chr']) &
        (exons_df['end'] >= row['start']) &
        (exons_df['start'] <= row['end'])
    ]

    total_overlap_length = 0
    total_exon_length = 0

    for _, exon in overlapping_exons.iterrows():
        # Calculate overlap
        overlap_start = max(row['start'], exon['start'])
        overlap_end = min(row['end'], exon['end'])
        overlap_length = max(0, overlap_end - overlap_start)

        # Add overlap length and exon length
        total_overlap_length += overlap_length
        total_exon_length += exon['end'] - exon['start']

    # Calculate overlap percentage
    if total_exon_length > 0:
        overlap_percentage = (total_overlap_length / total_exon_length) * 100
    else:
        overlap_percentage = 0

    return (total_overlap_length, overlap_percentage)

# Modified check_recalled function with pre-filtered exons
def check_recalled(row):
    estimated_cn = row['Estimated CN of CNV']
    cn = row['cn']
    VS_chr = row['VS_Call_chr']
    VS_start = row['VS_Call_start']
    VS_end = row['VS_Call_end']
    sim_chr = row['chr']
    sim_start = row['start']
    sim_end = row['end']
    count_exons_overlapping_simulated_window = row['count_exons_overlapping_simulated_window']
    count_exons_within_simulated_window = row['count_exons_within_simulated_window']
    overlap = row['VarSeq_overlap_within_window_based_on_window_length']

    # Count exons within the VS-called region based on pre-filtered simulated region
    count_sim_exons_within_VS_call = counting_sim_exons_within_VS_call(VS_chr, VS_start, VS_end, sim_chr, sim_start, sim_end, exons)
    count_all_exons_overlapping_VS_call = counting_all_exons_overlapping_VS_call(VS_chr, VS_start, VS_end, exons)
    
    # Define valid CN matches
    valid_cn_match = estimated_cn == cn or (estimated_cn, cn) in [(3, 4), (4, 3)]

    # Check recall conditions
    if cn != 2 and count_exons_overlapping_simulated_window > 0:
        if overlap > 0 and valid_cn_match:
            return (True, count_sim_exons_within_VS_call, count_all_exons_overlapping_VS_call)
        # if count_exons_within_simulated_window <= 1 and overlap > 0 and valid_cn_match:
        #     return (True, count_sim_exons_within_VS_call)
        # elif count_exons_within_simulated_window > 1 and count_sim_exons_within_VS_call > 0 and valid_cn_match:
        #     return (True, count_sim_exons_within_VS_call)

    return (False, count_sim_exons_within_VS_call, count_all_exons_overlapping_VS_call)

# Apply the function to each row and split the output into two separate columns
VarSeq_results_df[['recalled', 'count_sim_exons_within_VS_call', 'count_all_exons_overlapping_simulated_window']] = \
VarSeq_results_df.apply(lambda row: check_recalled(row), axis=1).apply(pd.Series)

VarSeq_results_df[['exon_overlap_length', 'exon_overlap_percentage']] = \
VarSeq_results_df.apply(lambda row: calculate_exon_overlap_percentage(row , exons_df=exons), axis=1).apply(pd.Series)

# # Apply calculate_exon_overlap_percentage functions
# unique_false_rows['exon_overlap_percentage'] = unique_false_rows.apply(
#     calculate_exon_overlap_percentage, exons_df=exons, axis=1
# )


# recalled['exon_overlap_percentage'] = recalled.apply(
#     calculate_overlap_percentage, exons_df=exons, axis=1
# )

In [21]:
len(VarSeq_results_df)

670

## Parse the 'Precision Level of call:' column to separate the comma-separated values.

In [22]:
VarSeq_results_df['Precision Level of call:'] = VarSeq_results_df['Precision Level of call:'].str.strip()

# Define the precision levels with a regular expression
precision_levels = {
    'Very High Sensitivity': r"\(1\) Very High Sensitivity",
    'High Sensitivity': r"\(2\) High Sensitivity",
    'Balanced': r"\(3\) Balanced",
    'High Precision': r"\(4\) High Precision",
    'Very High Precision': r"\(5\) Very High Precision"
}

# Create new columns with True/False based on presence of each precision level
for level_name, pattern in precision_levels.items():
    VarSeq_results_df[level_name] = VarSeq_results_df['Precision Level of call:'].str.contains(pattern)

## Extracting the false positives first

In [23]:
false_positives_varseq = VarSeq_results_df[VarSeq_results_df['VarSeq_overlap_within_window_based_on_window_length'] == 0]

In [24]:
print('number of false positives: ' + str(len(false_positives_varseq)))

number of false positives: 30


In [25]:
false_positives_varseq.to_csv(varseq_tables_path + '/false_positives_varseq.csv')

## Filtering out the five duplicate ClinVar deletions that i accidentally left in the simulations

In [26]:
# Identify groups where 'chr', 'start', 'end', 'gene', 'cn' have multiple unique 'sample' values
duplicated_groups = VarSeq_results_df.groupby(['chr', 'start', 'end', 'gene', 'cn'])['sample'].nunique()
filtered_keys = duplicated_groups[duplicated_groups > 1].index

# Split the dataframe into duplicates and non-duplicates
duplicates = VarSeq_results_df[VarSeq_results_df.set_index(['chr', 'start', 'end', 'gene', 'cn']).index.isin(filtered_keys)]
non_duplicates = VarSeq_results_df[~VarSeq_results_df.set_index(['chr', 'start', 'end', 'gene', 'cn']).index.isin(filtered_keys)]

# For duplicates, keep only the row where recalled == True, if it exists
duplicates = duplicates.sort_values(by=['recalled'], ascending=False).drop_duplicates(subset=['chr', 'start', 'end', 'gene', 'cn'], keep='first')

# Combine back the filtered duplicates with the original non-duplicates
VarSeq_results_df = pd.concat([non_duplicates, duplicates]).reset_index(drop=True)

In [27]:
duplicates

,cn,sample,chr,start,end,gene,count_exons_overlapping_simulated_window,count_exons_within_simulated_window,summed_length_exons_overlapping,summed_length_exons_within,...,recalled,count_sim_exons_within_VS_call,count_all_exons_overlapping_simulated_window,exon_overlap_length,exon_overlap_percentage,Very High Sensitivity,High Sensitivity,Balanced,High Precision,Very High Precision
348,1,cn1_DNS-XTHS-0106-C05-DNS009868_S131_chr17_430...,chr17,43047704,43051062,BRCA1,3,1,340,124,...,True,1,3,173.0,50.882353,True,True,True,True,True
395,1,cn1_DNS-XTHS-0106-E01-DNS009838_S101_chr17_430...,chr17,43067608,43067695,BRCA1,1,0,138,0,...,True,0,1,87.0,63.043478,True,True,True,True,True
411,1,cn1_DNS-XTHS-0106-E12-DNS009926_S189_chr17_430...,chr17,43076488,43076614,BRCA1,1,0,177,0,...,True,0,1,126.0,71.186441,True,True,True,True,True
455,1,cn1_DNS-XTHS-0105-H11-DNS009825_S88_chr17_4311...,chr17,43115726,43115779,BRCA1,1,0,104,0,...,True,0,1,53.0,50.961538,True,True,True,True,True
460,1,cn1_DNS-XTHS-0107-D06-DNS009973_S236_chr16_236...,chr16,23603162,23603669,PALB2,1,0,261,0,...,True,0,1,236.0,90.421456,True,True,False,False,False


## Printing out overall simulation results

In [28]:
# Print out number of CNVs not covered by panel bed file
print("Number of CNVs not covered by Panel BED file: ", len(VarSeq_results_df[VarSeq_results_df['count_exons_overlapping_simulated_window'] == 0]))

# Filter out rows where CNVs are not covered by the exon bed file:
VarSeq_results_df = VarSeq_results_df[VarSeq_results_df['count_exons_overlapping_simulated_window'] != 0]

Number of CNVs not covered by Panel BED file:  0


In [29]:
# Step 1: Group by the combination of 'sample', 'chr', 'start', 'end' and check if any of them have 'recalled' == True
grouped = VarSeq_results_df.groupby(['cn', 'sample', 'chr', 'start', 'end'])['recalled'].any().reset_index()

# Step 2: Filter out combinations where 'recalled' == True is present in the group
only_true_combinations = grouped[grouped['recalled'] == True][['cn', 'sample', 'chr', 'start', 'end']]

In [30]:
print('number of recalled CNVs: ' + str(len(only_true_combinations)))

number of recalled CNVs: 578


In [31]:
recalled_del = only_true_combinations[only_true_combinations['cn'] == 1]
print('number of recalled deletions: ' + str(len(recalled_del)))

number of recalled deletions: 504


In [32]:
recalled_dup = only_true_combinations[only_true_combinations['cn'] == 3]
print('number of recalled duplications: ' + str(len(recalled_dup)))

number of recalled duplications: 74


In [33]:
for level in precision_levels:
    df_slice = VarSeq_results_df[(VarSeq_results_df[level] == True) & (VarSeq_results_df['recalled'] == True)]
    df_slice_grouped = df_slice.groupby(['sample', 'cn', 'chr', 'start', 'end']).any().reset_index()
    
    print(f"Number of CNV calls recalled with {level}: {len(df_slice_grouped)}")
    break

Number of CNV calls recalled with Very High Sensitivity: 578


## Below is a CNV that is NOT called at Very High Sensitivity but called at lower sensitivity settings...

In [34]:
# Find rows in only_true_combinations where 'sample' is not in df_slice
diff = only_true_combinations[~only_true_combinations['sample'].isin(df_slice['sample'])]

print(diff)

Empty DataFrame
Columns: [cn, sample, chr, start, end]
Index: []


In [35]:
for level in precision_levels:
    df_slice = VarSeq_results_df[(VarSeq_results_df[level] == True) & (VarSeq_results_df['recalled'] == True)]
    df_slice_grouped = df_slice.groupby(['cn', 'sample', 'chr', 'start', 'end']).any().reset_index()
    
    print(f"Number of CNV calls recalled with {level}: {len(df_slice_grouped)}")

Number of CNV calls recalled with Very High Sensitivity: 578
Number of CNV calls recalled with High Sensitivity: 550
Number of CNV calls recalled with Balanced: 494
Number of CNV calls recalled with High Precision: 479
Number of CNV calls recalled with Very High Precision: 465


Note that some simulated CNVs have multiple calls made at different precision levels. The calls just have different coordinates (longer/shorter compared to the others).

If a call is made at multiple precision levels, the call made at 'higher' precision is usually more accurate to the intended regions that i simulate compared to the 'higher' sensitivity one.

In [36]:
VarSeq_results_df.to_csv(sim_dir + '/run' + dnascreen_run + '/plot_generation/Full_VS_CNV_Calls_metrics' + sensitivity + '.csv')

## Filtering out some CNV calls that are duplicate (simulated on the same region and sample) but got extra exons called
## also collapsing the duplicate calls to the call made at the higher precision.
## because i can't have duplicate CNV calls for the same simulation cause i'm feeding them to the R scripts for volcano plots

In [37]:
# VarSeq_results_df = VarSeq_results_df.sort_values(
#     by=['sample', 'recalled', 'Precision Level of call:'], 
#     ascending=[True, False, False]  # Sort 'Precision Level of call:' in descending order
# )

# VarSeq_results_df = VarSeq_results_df.drop_duplicates(subset=['sample', 'cn'], keep='first')

# VarSeq_results_df = VarSeq_results_df.reset_index(drop=True)

In [38]:
# Step 1: Filter rows where 'recalled' == True
filtered_df = VarSeq_results_df[VarSeq_results_df['recalled'] == True].copy()

# Step 2: Define a function to determine the best CNV per sample
def choose_best_cnv(group):
    # First, prioritize rows where 'Number of Exons by VarSeq call' >= 'count_exons_overlapping_simulated_window'
    group['exon_difference'] = group['Number of Exons by VarSeq call'] - group['count_exons_overlapping_simulated_window']
    
    # Prioritize CNVs where exon_difference is >= 0
    best_cnv = group[group['exon_difference'] >= 0]
    
    if not best_cnv.empty:
        # If multiple CNVs qualify, select the one with the smallest exon_difference
        return best_cnv.loc[best_cnv['exon_difference'].idxmin()]
    
    # If no CNVs qualify, choose the one with the smallest absolute exon_difference
    return group.loc[group['exon_difference'].abs().idxmin()]

# Step 3: Apply the function group-wise based on the 'sample' column
filtered_df = filtered_df.groupby('sample', group_keys=False).apply(choose_best_cnv)

# Step 4: Drop the temporary 'exon_difference' column
filtered_df = filtered_df.drop(columns=['exon_difference'])

# Result: DataFrame with only the best CNV per sample
filtered_df

,cn,sample,chr,start,end,gene,count_exons_overlapping_simulated_window,count_exons_within_simulated_window,summed_length_exons_overlapping,summed_length_exons_within,...,recalled,count_sim_exons_within_VS_call,count_all_exons_overlapping_simulated_window,exon_overlap_length,exon_overlap_percentage,Very High Sensitivity,High Sensitivity,Balanced,High Precision,Very High Precision
sample,,,,,,,,,,,,,,,,,,,,,
cn1_DNS-XTHS-0105-A01-DNS009738_S1_chr17_43124017_43124115.sorted,1,cn1_DNS-XTHS-0105-A01-DNS009738_S1_chr17_43124...,chr17,43124017,43124115,BRCA1,1,0,130,0,...,True,0,1,98.0,75.384615,True,True,True,True,True
cn1_DNS-XTHS-0105-A01-DNS009738_S1_chr2_47403295_47407170.sorted,1,cn1_DNS-XTHS-0105-A01-DNS009738_S1_chr2_474032...,chr2,47403295,47407170,MSH2,1,0,261,0,...,True,0,1,132.0,50.574713,True,True,False,False,False
cn1_DNS-XTHS-0105-A02-DNS009746_S9_chr17_43045677_43049120.sorted,1,cn1_DNS-XTHS-0105-A02-DNS009746_S9_chr17_43045...,chr17,43045677,43049120,BRCA1,3,1,410,111,...,True,1,3,286.0,69.756098,True,True,False,False,False
cn1_DNS-XTHS-0105-A02-DNS009746_S9_chr19_11100222_11107515.sorted,1,cn1_DNS-XTHS-0105-A02-DNS009746_S9_chr19_11100...,chr19,11100222,11107515,LDLR,5,3,1123,777,...,True,3,5,1074.0,95.636687,False,True,True,True,True
cn1_DNS-XTHS-0105-A03-DNS009754_S17_chr17_43102513_43107537.sorted,1,cn1_DNS-XTHS-0105-A03-DNS009754_S17_chr17_4310...,chr17,43102513,43107537,BRCA1,3,3,457,457,...,True,3,3,457.0,100.000000,True,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
cn3_DNS-XTHS-0107-G06-DNS010250_S239_chr3_37011814_37042337.sorted,3,cn3_DNS-XTHS-0107-G06-DNS010250_S239_chr3_3701...,chr3,37011814,37042337,MLH1,9,7,1636,1429,...,True,7,9,1597.0,97.616137,True,True,True,True,True
cn3_DNS-XTHS-0107-G08-DNS009992_S255_chr19_11116092_11116999.sorted,3,cn3_DNS-XTHS-0107-G08-DNS009992_S255_chr19_111...,chr19,11116092,11116999,LDLR,2,0,359,0,...,True,0,2,311.0,86.629526,True,True,True,True,False
cn3_DNS-XTHS-0107-G10-DNS010008_S271_chr17_43099070_43102393.sorted,3,cn3_DNS-XTHS-0107-G10-DNS010008_S271_chr17_430...,chr17,43099070,43102393,BRCA1,1,1,156,156,...,True,1,1,156.0,100.000000,True,True,True,True,True


In [39]:
len(filtered_df) == len(only_true_combinations)

True

In [40]:
recalled = filtered_df

## NOTE THAT THE Recalled_VS_CNV_Calls_metrics contains calls with highest precision only.

In [41]:
recalled.to_csv(sim_dir + '/run' + dnascreen_run + '/plot_generation/Recalled_VS_CNV_Calls_metrics' + sensitivity + '.csv')

## Recalled CNVs that do not have all exons within CNV that are detected

## Good to have this as a sanity check. NOTE: One CNV in Run 33 have one less exon within CNV that is not detected as part of the CNV

In [42]:
recalled_cnvs_not_all_exons_called = \
filtered_df[(filtered_df['recalled'] == True) & (filtered_df['count_exons_within_simulated_window'] > filtered_df['count_sim_exons_within_VS_call'])]

In [43]:
print('number of recalled CNVs that do not have all exons within CNV detected: ' + str(len(recalled_cnvs_not_all_exons_called)))

number of recalled CNVs that do not have all exons within CNV detected: 0


In [44]:
recalled_cnvs_not_all_exons_called.to_csv(sim_dir + '/run' + dnascreen_run + '/plot_generation/recalled_cnvs_not_all_exons_called.csv')

## Listing down missed CNVs by VarSeq

In [45]:
# Step 1: Group by the combination of 'sample', 'chr', 'start', 'end' and check if any of them have 'recalled' == True
grouped = VarSeq_results_df.groupby(['cn', 'sample', 'chr', 'start', 'end'])['recalled'].any().reset_index()

# Step 2: Filter out combinations where 'recalled' == True is present in the group
only_false_combinations = grouped[(grouped['recalled'] == False) & (grouped['cn'] != 2)][['cn', 'sample', 'chr', 'start', 'end']]

# Step 3: Merge back with the original DataFrame to get only rows with 'recalled' == False
unique_false_rows = VarSeq_results_df.merge(only_false_combinations, on=['cn', 'sample', 'chr', 'start', 'end'])
unique_false_rows = unique_false_rows[unique_false_rows['recalled'] == False]

In [46]:
print('number of missed CNVs: ' + str(len(only_false_combinations)))

number of missed CNVs: 43


In [47]:
missed_del = only_false_combinations[only_false_combinations['cn'] == 1]
print('number of missed deletions: ' + str(len(missed_del)))
print('number of recalled deletions: ' + str(533 - len(missed_del)))

number of missed deletions: 29
number of recalled deletions: 504


In [48]:
missed_dup = only_false_combinations[only_false_combinations['cn'] == 3]
print('number of missed duplications: ' + str(len(missed_dup)))
print('number of recalled duplications: ' + str(88 - len(missed_dup)))

number of missed duplications: 14
number of recalled duplications: 74


In [49]:
unique_false_rows.to_csv(varseq_tables_path + '/missed_cnvs_with_varseq_calls.csv')

In [50]:
# For easy exporting as a table to be presented
unique_false_rows[['cn', 'chr', 'start', 'end', 'gene', 'summed_length_exons_overlapping', \
                   'Percent Difference', 'Read Ratio of Simulated CNV', 'Read Ratio of Original CNV region', 'Z-Score of Simulated CNV', 'Z-Score of Original CNV region', \
                   'Target Mean Depth of Original CNV region', 'Target Mean Depth of Simulated CNV']].to_csv(varseq_tables_path + '/notion_missed_cnvs_with_varseq_calls.csv')

## Extracting all Z-Scores of non-overlapping exons to plot mean and std error bars in Z-Score vs CN plot later

In [51]:
# # Create a list to store the flattened Z-scores and their corresponding CN
# z_scores_flattened = []
# cn_values = []

# for cn, z_scores in enumerate(avg_z_scores_all_nonmatching_sim_exons):
#     z_scores_flattened.extend(z_scores)
#     cn_values.extend([cn] * len(z_scores))

# # Create a DataFrame from the flattened list
# z_scores_per_cn_nonmatching_sim_exons = pd.DataFrame({
#     'z_score_varseq': z_scores_flattened,
#     'cn': cn_values
# })


In [52]:
# z_scores_per_cn_nonmatching_sim_exons.to_csv(sim_dir + '/run' + dnascreen_run + '/plot_generation/z_scores_per_cn_nonmatching_exons.csv')

## Storing per sample CNV calls for Exome Depth
## Merge rows with calls split between 10 runs into a single row. Separate the metrics from the different calls by comma in some columns

In [53]:
# # Importing Exome Depth calls
# df_ED_CNV = pd.read_csv(sim_dir + '/run' + dnascreen_run + '/plot_generation/exome_depth_stats_different_ref_samples.csv')

In [54]:
# # Define the columns to concatenate
# columns_to_concat = [
#     'Frequency', 'ED_length', 'ED_chr', 'ED_start', 'ED_end', 'ED_percentage_within_window',
#     'Mean_Reads_Expected', 'Mean_Reads_Observed', 'Mean_Reads_Ratio', 'Mean_BF', 'Mean_Correlation'
# ]

# # Function to concatenate values for the specified columns
# def concatenate_values(group):
#     concatenated_values = {col: ', '.join(group[col].astype(str)) for col in columns_to_concat}
#     return pd.Series(concatenated_values)


In [55]:
# # Group by 'sample' and apply the concatenation function
# merged_df_ED_CNV = df_ED_CNV.groupby('sample').apply(concatenate_values).reset_index()

# # Drop duplicate rows and merge with concatenated values
# df_ED_CNV = df_ED_CNV.drop(columns=columns_to_concat).drop_duplicates(subset='sample')
# df_ED_CNV_final = pd.merge(df_ED_CNV, merged_df_ED_CNV, on='sample')

In [56]:
# df_ED_CNV_final

In [57]:
# merged_df = varseq_results_df_final.merge(df_ED_CNV_final, left_on=['sample', 'chr', 'start', 'end'], right_on=['sample', 'sim_chr', 'sim_start', 'sim_end'])

In [58]:
# # clean up the table by removing unecessary columns
# columns_to_drop = ['overlap_with_start_chr', 'overlap_with_start_start', 
#                    'overlap_with_start_end', 'overlap_with_start_name', 
#                    'overlap_with_end_chr', 'overlap_with_end_start', 
#                    'overlap_with_end_end', 'overlap_with_end_name',
#                    'call_key', 'sim_chr', 'sim_start', 
#                    'sim_end', 'sim_length', 'cn_ED'
#                   ]

# merged_df.drop(columns=columns_to_drop, inplace=True)

In [59]:
# merged_df.to_csv(sim_dir + '/run' + dnascreen_run + '/plot_generation/VS_ED_CNV_Calls_metrics.csv')